# TUẦN 1: 2.1 + 2.2

In [ ]:
# Cài Java 11 (khuyến nghị cho Spark 3.5.x)
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("Java version:")
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Java version:
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
# Cách 2: Tải Spark binary thủ công
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz

import os
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
!pip install findspark -q
import findspark
findspark.init()

In [ ]:
# Khởi tạo SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Olist_Preprocessing_Week1_Final") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("Spark Session đã khởi tạo thành công!")

Spark Session đã khởi tạo thành công!


In [ ]:
!pip install -q streamlit
print("\nStreamlit installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 53.5 MB/s eta 0:00:00

Streamlit installed successfully.


In [ ]:
!python --version

Python 3.12.13


In [ ]:
!pip install -q gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/datasetCuoiKi/data/"

In [ ]:
# Đọc dữ liệu
orders_df = spark.read.csv(base_path + "olist_orders_dataset.csv", header=True, inferSchema=True)
customers_df = spark.read.csv(base_path + "olist_customers_dataset.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv(base_path + "olist_order_items_dataset.csv", header=True, inferSchema=True)
payments_df = spark.read.csv(base_path + "olist_order_payments_dataset.csv", header=True, inferSchema=True)
reviews_df = spark.read.csv(base_path + "olist_order_reviews_dataset.csv", header=True, inferSchema=True)
products_df = spark.read.csv(base_path + "olist_products_dataset.csv", header=True, inferSchema=True)
sellers_df = spark.read.csv(base_path + "olist_sellers_dataset.csv", header=True, inferSchema=True)
category_df = spark.read.csv(base_path + "product_category_name_translation.csv", header=True, inferSchema=True)
geolocation_df = spark.read.csv(base_path + "olist_geolocation_dataset.csv", header=True, inferSchema=True)
print("Đã đọc xong 9 file CSV")

Đã đọc xong 9 file CSV


In [ ]:
from pyspark.sql.functions import col, sum

def explore_full(df, name):
    print("\n==============================")
    print("DATASET:", name)
    print("==============================")

    # 1. Số dòng
    total_rows = df.count()
    print("Số dòng:", total_rows)

    # 2. Schema
    print("\nSchema:")
    df.printSchema()

    # 3. Dữ liệu mẫu
    print("\n5 dòng đầu:")
    df.show(5, truncate=False)

    # 4. Thống kê mô tả
    print("\nThống kê:")
    df.describe().show()

    # 5. Kiểm tra null
    print("\nSố lượng NULL theo cột:")
    null_df = df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    null_df.show()

    # 6. Kiểm tra duplicate
    unique_rows = df.dropDuplicates().count()
    print("\nDuplicate:")
    print("Tổng:", total_rows)
    print("Unique:", unique_rows)
    print("Số dòng trùng:", total_rows - unique_rows)

In [ ]:
explore_full(customers_df, "customers")


DATASET: customers
Số dòng: 99441

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29

In [ ]:
explore_full(geolocation_df, "geolocation")


DATASET: geolocation
Số dòng: 1000163

Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)


5 dòng đầu:
+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng   |geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|1037                       |-23.54562128115268 |-46.63929204800168|sao paulo       |SP               |
|1046                       |-23.546081127035535|-46.64482029837157|sao paulo       |SP               |
|1046                       |-23.54612896641469 |-46.64295148361138|sao paulo       |SP               |
|1041                       |-23.5443921648681  |-46.63949

In [ ]:
explore_full(order_items_df, "order_items")


DATASET: order_items
Số dòng: 112650

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


5 dòng đầu:
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77

In [ ]:
explore_full(payments_df, "payments")


DATASET: payments
Số dòng: 103886

Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)


5 dòng đầu:
+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.7

In [ ]:
explore_full(reviews_df, "reviews")


DATASET: reviews
Số dòng: 104162

Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+---

In [ ]:
explore_full(orders_df, "orders")


DATASET: orders
Số dòng: 99441

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+-

In [ ]:
explore_full(products_df, "products")


DATASET: products
Số dòng: 32951

Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)


5 dòng đầu:
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+---------------------

In [ ]:
explore_full(sellers_df, "sellers")


DATASET: sellers
Số dòng: 3095

Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)


5 dòng đầu:
+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+-----------

In [ ]:
explore_full(category_df, "category_translation")


DATASET: category_translation
Số dòng: 71

Schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)


5 dòng đầu:
+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
+----------------------+-----------------------------+
only showing top 5 rows


Thống kê:
+-------+---------------------+-----------------------------+
|summary|product_category_name|product_category_name_english|
+-------+---------------------+-----------------------------+
|  count|                   71|                           71|
|   mean|                 NULL|     

In [ ]:
from pyspark.sql.functions import *

In [ ]:
# Geolocation
geo_customers = geolocation_df.select(
    col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
    col("geolocation_city").alias("customer_geo_city"),
    col("geolocation_state").alias("customer_geo_state")
).dropDuplicates()

customers_df = customers_df.join(geo_customers, "customer_zip_code_prefix", "left")

geo_sellers = geolocation_df.select(
    col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
    col("geolocation_city").alias("seller_geo_city"),
    col("geolocation_state").alias("seller_geo_state")
).dropDuplicates()

sellers_df = sellers_df.join(geo_sellers, "seller_zip_code_prefix", "left")

In [ ]:
# Timestamp & Cast
orders_df = orders_df \
    .withColumn("order_purchase_timestamp", try_to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", try_to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", try_to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", try_to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", try_to_timestamp(col("order_estimated_delivery_date")))

reviews_df = reviews_df \
    .withColumn("review_creation_date", try_to_timestamp(col("review_creation_date"))) \
    .withColumn("review_answer_timestamp", try_to_timestamp(col("review_answer_timestamp")))

order_items_df = order_items_df \
    .withColumn("shipping_limit_date", try_to_timestamp(col("shipping_limit_date"))) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments_df = payments_df.withColumn("payment_value", col("payment_value").cast("double"))

from pyspark.sql.functions import expr

reviews_df = reviews_df \
    .withColumn("review_score", expr("try_cast(review_score as int)"))
#reviews_df = reviews_df.withColumn("review_score", col("review_score").cast("int"))

In [ ]:
# Aggregate
order_items_agg = order_items_df.groupBy("order_id").agg(
    first("product_id").alias("product_id"),
    first("seller_id").alias("seller_id"),
    sum("price").alias("total_price"),
    sum("freight_value").alias("total_freight_value"),
    count("order_item_id").alias("num_items")
)

payments_agg = payments_df.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    first("payment_type").alias("payment_type")
)

In [ ]:
# JOIN
df_master = orders_df.alias("o") \
    .join(customers_df.alias("c"), col("o.customer_id") == col("c.customer_id"), "left") \
    .drop(col("c.customer_id")) \
    .join(order_items_agg.alias("oi"), col("o.order_id") == col("oi.order_id"), "left") \
    .drop(col("oi.order_id")) \
    .join(payments_agg.alias("op"), col("o.order_id") == col("op.order_id"), "left") \
    .drop(col("op.order_id")) \
    .join(reviews_df.alias("r"), col("o.order_id") == col("r.order_id"), "left") \
    .drop(col("r.order_id")) \
    .join(products_df.alias("p"), col("oi.product_id") == col("p.product_id"), "left") \
    .drop(col("p.product_id")) \
    .join(sellers_df.alias("s"), col("oi.seller_id") == col("s.seller_id"), "left") \
    .drop(col("s.seller_id")) \
    .join(category_df.alias("t"), col("p.product_category_name") == col("t.product_category_name"), "left") \
    .drop(col("t.product_category_name"))

In [ ]:
# ================================
# XỬ LÝ MISSING VALUES (FINAL) - ĐÃ SỬA
# ================================

# 1. Tạo cột delivery_delay_days trước (nếu chưa có)
from pyspark.sql.functions import datediff, col

df_master = df_master.withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

# 2. Xử lý missing values
df_master = df_master.fillna({
    # TEXT
    "review_comment_message": "",
    "review_comment_title": "",

    # CATEGORICAL
    "payment_type": "unknown",
    "product_category_name_english": "unknown",

    # NUMERICAL
    "delivery_delay_days": 0,      # giờ đã tồn tại
    "num_items": 1,

    # NUMERICAL (bổ sung)
    "total_payment_value": 0,
    "total_price": 0,
    "total_freight_value": 0
})

# Drop rows thiếu key quan trọng
df_master = df_master.dropna(subset=["order_id"])

print("✅ Missing values đã xử lý đầy đủ")

✅ Missing values đã xử lý đầy đủ


In [ ]:
# Feature Engineering
df_master = df_master \
    .withColumn("delivery_delay_days", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))) \
    .withColumn("total_order_value", col("total_price") + col("total_freight_value")) \
    .withColumn("is_late_delivery", when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0))

# ÉP 1 DÒNG CHO 1 ORDER_ID
df_master = df_master.dropDuplicates(["order_id"])

# THANH HUYỀN

# ALS_MODEL

**TẠO DATASET CHUẨN CHO ALS**

In [ ]:
from pyspark.sql.functions import col

# Dataset chuẩn cho ALS
als_df = orders_df \
    .join(order_items_df, "order_id") \
    .join(reviews_df, "order_id") \
    .select(
        col("customer_id"),
        col("product_id"),
        col("review_score")
    ) \
    .dropna()

als_df.show(5)

+--------------------+--------------------+------------+
|         customer_id|          product_id|review_score|
+--------------------+--------------------+------------+
|9ef432eb625129730...|87285b34884572647...|           4|
|b0830fb4747a6c6d2...|595fac2a385ac33a8...|           4|
|41ce2a54c0b03bf34...|aa4383b373c6aca5d...|           5|
|f88197465ea7920ad...|d0b61bfb1de832b15...|           5|
|8ab97904e6daea886...|65266b2da20d04dbe...|           5|
+--------------------+--------------------+------------+
only showing top 5 rows



**ENCODE USER & ITEM**

In [ ]:
from pyspark.ml.feature import StringIndexer

user_indexer = StringIndexer(inputCol="customer_id", outputCol="user_idx", handleInvalid="keep")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_idx", handleInvalid="keep")

# lưu model
user_indexer_model = user_indexer.fit(als_df)
item_indexer_model = item_indexer.fit(als_df)

als_df = user_indexer_model.transform(als_df)
als_df = item_indexer_model.transform(als_df)
#  Lưu labels để mapping sau
item_labels = item_indexer_model.labels
# giữ lại product_id để mapping sau
als_df = als_df.select("user_idx", "item_idx", "review_score", "product_id")

**TRAIN / TEST SPLIT**

In [ ]:
train_df, test_df = als_df.randomSplit([0.8, 0.2], seed=42)

**TRAIN ALS MODEL**

In [ ]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_idx",
    itemCol="item_idx",
    ratingCol="review_score",
    rank=20,
    maxIter=15,
    regParam=0.1,
    implicitPrefs=False,
    coldStartStrategy="drop"
)

als_model = als.fit(train_df)

**ĐÁNH GIÁ RMSE**

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = als_model.transform(test_df)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="review_score",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print("RMSE =", rmse)

RMSE = 1.500445115089628


**TOP-N RECOMMENDATION**

In [ ]:
user_recs = als_model.recommendForAllUsers(10)
user_recs.show(5, truncate=False)

+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_idx|recommendations                                                                                                                                                                                      |
+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1       |[{2306, 2.2085574}, {5735, 2.0474956}, {10026, 1.8614056}, {6316, 1.8469743}, {13268, 1.8258202}, {18762, 1.7379798}, {23459, 1.7228361}, {11797, 1.7208734}, {15725, 1.7090425}, {21776, 1.7007065}]|
|3       |[{681, 4.9325423}, {21505, 3.730084}, {1642, 3.7193308}, {23881, 3.6569257}, {8919, 3.4909637}, {16936, 3.4314659}, {21618, 3.3429074}, {14969, 3.341002},

**EXPLODE KẾT QUẢ**

In [ ]:
from pyspark.sql.functions import explode, col

user_recs_exploded = user_recs \
    .withColumn("rec", explode("recommendations")) \
    .select(
        "user_idx",
        col("rec.item_idx").alias("item_idx"),
        col("rec.rating").alias("score")
    )

user_recs_exploded.show(5)

+--------+--------+---------+
|user_idx|item_idx|    score|
+--------+--------+---------+
|       1|    2306|2.2085574|
|       1|    5735|2.0474956|
|       1|   10026|1.8614056|
|       1|    6316|1.8469743|
|       1|   13268|1.8258202|
+--------+--------+---------+
only showing top 5 rows



**MAP item_idx → product_id**

In [ ]:
# tạo bảng mapping
mapping_df = spark.createDataFrame(
    [(i, item_labels[i]) for i in range(len(item_labels))],
    ["item_idx", "product_id"]
)

# join mapping
final_recs = user_recs_exploded.join(mapping_df, "item_idx", "left")

final_recs.show(5)

+--------+--------+---------+--------------------+
|item_idx|user_idx|    score|          product_id|
+--------+--------+---------+--------------------+
|   13268|       1|1.8258202|bbc3a3f2596e84954...|
|    5735|       1|2.0474956|957bdadbdfc3aa7b3...|
|    2306|       1|2.2085574|4a8c87facc0a20b72...|
|   18762|       1|1.7379798|36c554eb2e204a1db...|
|   10026|       1|1.8614056|2bb7ee4eb5d38207b...|
+--------+--------+---------+--------------------+
only showing top 5 rows



**JOIN THÔNG TIN SẢN PHẨM**

In [ ]:
final_recs = final_recs.join(
    products_df.select("product_id", "product_category_name"),
    on="product_id",
    how="left"
)

final_recs.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- item_idx: integer (nullable = true)
 |-- user_idx: integer (nullable = false)
 |-- score: float (nullable = true)
 |-- product_category_name: string (nullable = true)



**DEMO USER**

In [ ]:
sample_user_id = als_df.select("user_idx").distinct().limit(1).collect()[0][0]

final_recs.filter(col("user_idx") == sample_user_id) \
    .orderBy(col("score").desc()) \
    .select(
        "user_idx",
        "product_id",
        "product_category_name",
        "score"
    ) \
    .show(truncate=False)

+--------+--------------------------------+---------------------+---------+
|user_idx|product_id                      |product_category_name|score    |
+--------+--------------------------------+---------------------+---------+
|64662   |87285b34884572647811a353c7ac498a|utilidades_domesticas|3.9207158|
|64662   |6dceb64b44142aeb9fe7397dfefae705|automotivo           |3.7245302|
|64662   |642369377615febc7fa89e4c8df5110e|beleza_saude         |3.4237041|
|64662   |c54ed8ea69e24c191fbc23b3234c6b32|beleza_saude         |3.3938882|
|64662   |9592ea6bf88d20db726179927bbefeb2|utilidades_domesticas|3.3608093|
|64662   |ffe8083298f95571b4a66bfbc1c05524|cama_mesa_banho      |3.27325  |
|64662   |72b781d37ad5c06da9a06f01248d3f48|cama_mesa_banho      |3.2732487|
|64662   |1e618d311a1b7f88a9d96ec50aa85582|utilidades_domesticas|3.2688158|
|64662   |ab481d9ef0fb9cab12778cdf56564ff7|fashion_calcados     |3.2610867|
|64662   |9f31b36609877849f75e9586f6b6a62c|fashion_calcados     |3.2610867|
+--------+--

**HIỂN THỊ TOÀN BỘ**

In [ ]:
final_recs.select(
    "user_idx",
    "product_id",
    "product_category_name",
    "score"
).show(10, truncate=False)

+--------+--------------------------------+---------------------+---------+
|user_idx|product_id                      |product_category_name|score    |
+--------+--------------------------------+---------------------+---------+
|1       |bbc3a3f2596e849548463e3a46d833f0|cool_stuff           |1.8258202|
|1       |0c5aecba8eab00eb055c6e2e7b9b4346|beleza_saude         |1.7090425|
|1       |957bdadbdfc3aa7b33ac86d926721838|beleza_saude         |2.0474956|
|1       |61d83b8d80a61b5fd688471b5b273b5f|perfumaria           |1.7007065|
|1       |7aac5971b0560a816c14f6b6e9185bcc|beleza_saude         |1.7208734|
|1       |4a8c87facc0a20b726d5e1112ff3069f|bebes                |2.2085574|
|1       |36c554eb2e204a1db6000ed146c9fbde|automotivo           |1.7379798|
|3       |89b190a046022486c635022524a974a8|moveis_decoracao     |4.9325423|
|1       |78d9cee7386ff7a3fb916ee1c076a278|automotivo           |1.7228361|
|1       |2bb7ee4eb5d38207b650bd2ff6f4586c|esporte_lazer        |1.8614056|
+--------+--

**COLD START**

In [ ]:
print("Cold-start strategy:", als.getColdStartStrategy())

Cold-start strategy: drop


**SAVE MODEL**

In [ ]:
als_model.save("/content/drive/MyDrive/datasetCuoiKi/data/als_model")

**ITEM-BASED (BONUS)**

In [ ]:
item_recs = als_model.recommendForAllItems(10)
item_recs.show(5, truncate=False)

+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|item_idx|recommendations                                                                                                                                                                                         |
+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1       |[{1439, 4.893411}, {1400, 4.893411}, {1168, 4.893411}, {38, 4.887513}, {95494, 4.8835926}, {94761, 4.8835926}, {92289, 4.8835926}, {92155, 4.8835926}, {89826, 4.8835926}, {89720, 4.8835926}]          |
|3       |[{97600, 4.8716593}, {97128, 4.8716593}, {96984, 4.8716593}, {96831, 4.8716593}, {95126, 4.8716593}, {94509, 4.8716593}, {94335, 4.8716593}, {

# FD_MODEL


**TẠO TRANSACTION DATASET**

In [ ]:
from pyspark.sql.functions import collect_set, col, size

# chỉ giữ transaction có ≥ 2 items
transactions_df = order_items_df \
    .groupBy("order_id") \
    .agg(collect_set("product_id").alias("items")) \
    .filter(size(col("items")) >= 2)

transactions_df.show(5, truncate=False)
print("Total transactions:", transactions_df.count())

+--------------------------------+------------------------------------------------------------------------------------------------------+
|order_id                        |items                                                                                                 |
+--------------------------------+------------------------------------------------------------------------------------------------------+
|00337fe25a3780b3424d9ad7c5a4b35e|[1f9799a175f50c9fa725984775cac5c5, 13944d17b257432717fd260e69853140]                                  |
|0097f0545a302aafa32782f1734ff71c|[b6397895a17ce86decd60b898b459796, 636598095d69a5718e67d2c9a3c7dde6]                                  |
|01235dc626dcf13283207ba7f36a959a|[176faef125e3675b4eeaa6083ada61e1, 10dae91e0aba95747e90280edeffe883]                                  |
|012a238ab54294a3b365812ccc82b135|[8338cef8355d238f43711dcb9c0657b2, 07c055536ebf10dfbb6c6db6dbfc36e5, 741a31499a578979be85db7f80139e62]|
|0132451f29a10b66a5cf1bacc85f9afe|

**TRAIN FP-GROWTH**

In [ ]:
from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.002,
    minConfidence=0.3
)

fp_model = fp.fit(transactions_df)

**FREQUENT ITEMSETS**

In [ ]:
freq_items = fp_model.freqItemsets

freq_items.orderBy(col("freq").desc()).show(10, truncate=False)
print("Total frequent itemsets:", freq_items.count())

+--------------------------------------------------------------------+----+
|items                                                               |freq|
+--------------------------------------------------------------------+----+
|[99a4788cb24856965c36a24e339b6058]                                  |52  |
|[36f60d45225e60c7da4558b070ce4b60]                                  |48  |
|[e53e557d5a159f5aa2c5e995dfdf244b]                                  |38  |
|[35afc973633aaeb6b877ff57b2793310]                                  |36  |
|[e53e557d5a159f5aa2c5e995dfdf244b, 36f60d45225e60c7da4558b070ce4b60]|34  |
|[422879e10f46682990de24d770e7f83d]                                  |32  |
|[53759a2ecddad2bb87a079a1f1519f73]                                  |29  |
|[35afc973633aaeb6b877ff57b2793310, 99a4788cb24856965c36a24e339b6058]|29  |
|[389d119b48cf3043d311335e499d9c6b]                                  |28  |
|[368c6c730842d78016ad823897a372db]                                  |27  |
+-----------

**ASSOCIATION RULES**

In [ ]:
rules = fp_model.associationRules

rules.show(10, truncate=False)
print("Total rules:", rules.count())

+----------------------------------+----------------------------------+-------------------+------------------+---------------------+
|antecedent                        |consequent                        |confidence         |lift              |support              |
+----------------------------------+----------------------------------+-------------------+------------------+---------------------+
|[99a4788cb24856965c36a24e339b6058]|[35afc973633aaeb6b877ff57b2793310]|0.5576923076923077 |50.13034188034188 |0.00896168108776267  |
|[4fcb3d9a5f4871e8362dfedbdb02b064]|[f4f67ccaece962d013a4e1d7dc3a61f7]|0.8947368421052632 |160.85380116959064|0.005253399258343634 |
|[e53e557d5a159f5aa2c5e995dfdf244b]|[36f60d45225e60c7da4558b070ce4b60]|0.8947368421052632 |60.32017543859649 |0.010506798516687269 |
|[389d119b48cf3043d311335e499d9c6b]|[53759a2ecddad2bb87a079a1f1519f73]|0.32142857142857145|35.86699507389163 |0.002781211372064277 |
|[389d119b48cf3043d311335e499d9c6b]|[422879e10f46682990de24d770e7f83d

**TÍNH LIFT**

In [ ]:
from pyspark.sql.functions import explode

# tổng số transaction
total_txn = transactions_df.count()

# support của từng item đơn
item_support = freq_items \
    .filter(size(col("items")) == 1) \
    .select(
        col("items")[0].alias("item"),
        (col("freq") / total_txn).alias("item_support")
    )

# explode consequent
rules_exploded = rules.withColumn("consequent_item", explode("consequent"))

# join để tính lift
rules_with_lift = rules_exploded.join(
    item_support,
    rules_exploded.consequent_item == item_support.item,
    "left"
).withColumn(
    "lift",
    col("confidence") / col("item_support")
)

rules_with_lift.select(
    "antecedent",
    "consequent",
    "support",
    "confidence",
    "lift"
).orderBy(col("lift").desc()).show(10, truncate=False)

+----------------------------------+----------------------------------+---------------------+-------------------+------------------+
|antecedent                        |consequent                        |support              |confidence         |lift              |
+----------------------------------+----------------------------------+---------------------+-------------------+------------------+
|[dbb67791e405873b259e4656bf971246]|[18486698933fbb64af6c0a255f7dd64c]|0.0021631644004944375|1.0                |462.2857142857143 |
|[18486698933fbb64af6c0a255f7dd64c]|[dbb67791e405873b259e4656bf971246]|0.0021631644004944375|1.0                |462.2857142857143 |
|[f4f67ccaece962d013a4e1d7dc3a61f7]|[4fcb3d9a5f4871e8362dfedbdb02b064]|0.005253399258343634 |0.9444444444444444 |160.85380116959064|
|[4fcb3d9a5f4871e8362dfedbdb02b064]|[f4f67ccaece962d013a4e1d7dc3a61f7]|0.005253399258343634 |0.8947368421052632 |160.85380116959064|
|[e53e557d5a159f5aa2c5e995dfdf244b]|[36f60d45225e60c7da4558b070ce4b60

**TOP RULES**

In [ ]:
top_rules = rules_with_lift \
    .orderBy(col("lift").desc()) \
    .limit(10)

top_rules.show(truncate=False)

+----------------------------------+----------------------------------+-------------------+------------------+---------------------+--------------------------------+--------------------------------+---------------------+
|antecedent                        |consequent                        |confidence         |lift              |support              |consequent_item                 |item                            |item_support         |
+----------------------------------+----------------------------------+-------------------+------------------+---------------------+--------------------------------+--------------------------------+---------------------+
|[dbb67791e405873b259e4656bf971246]|[18486698933fbb64af6c0a255f7dd64c]|1.0                |462.2857142857143 |0.0021631644004944375|18486698933fbb64af6c0a255f7dd64c|18486698933fbb64af6c0a255f7dd64c|0.0021631644004944375|
|[18486698933fbb64af6c0a255f7dd64c]|[dbb67791e405873b259e4656bf971246]|1.0                |462.2857142857143 |0.0021

**FILTER RULES**

In [ ]:
strong_rules = rules_with_lift.filter(
    (col("confidence") > 0.5) & (col("lift") > 1.2)
)

strong_rules.show(10, truncate=False)

+----------------------------------+----------------------------------+------------------+------------------+---------------------+--------------------------------+--------------------------------+---------------------+
|antecedent                        |consequent                        |confidence        |lift              |support              |consequent_item                 |item                            |item_support         |
+----------------------------------+----------------------------------+------------------+------------------+---------------------+--------------------------------+--------------------------------+---------------------+
|[e53e557d5a159f5aa2c5e995dfdf244b]|[36f60d45225e60c7da4558b070ce4b60]|0.8947368421052632|60.32017543859649 |0.010506798516687269 |36f60d45225e60c7da4558b070ce4b60|36f60d45225e60c7da4558b070ce4b60|0.014833127317676144 |
|[99a4788cb24856965c36a24e339b6058]|[35afc973633aaeb6b877ff57b2793310]|0.5576923076923077|50.13034188034188 |0.008961681

SAVE MODEL

In [ ]:
fp_model.save("/content/drive/MyDrive/datasetCuoiKi/data/fp_model")